# RAASTA — train day and night from public photos

**Use this notebook.** Upload this file to Colab (File → Upload notebook). Do not keep an old copy open.

You do **not** film any roads. This notebook downloads public photos, mixes them with your old training set, then trains the same **11 classes** again.

**How to run**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run cells from the top, **one download at a time**
3. If a download cell errors, skip it and run the next one
4. After training, download the `.tflite` and replace the file in the app

If Colab disconnects, re-run **Setup**, then the mix cell, then train or resume.

**What gets added**
- India road photos: cracks and potholes (day)
- Extra pothole photos (day)
- Dark photos of people, dogs, and cats (night)
- Buffalo, horse, goat, and donkey photos with boxes
- About 2000 photos from your old `merged` folder, so speed bumps and the other animals are not forgotten

**Training:** 30 epochs, image size 640, starting from your latest `best.pt`. About 2–4 hours on a T4.

Class order stays the same as the phone:
`pothole, crack, speed_bump, person, cow, buffalo, dog, cat, horse, donkey, goat`

Folders on Drive:
```
MyDrive/raasta/
  raw/        zips
  extracted/  unzipped photos
  merged/     your old YOLO set (already there)
  weights/    new training run
  exports/    the phone file
```

## 0) Setup — mount Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, zipfile, subprocess

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/raasta')
RAW = ROOT / 'raw'
EXTRACTED = ROOT / 'extracted'
MERGED = ROOT / 'merged'
WEIGHTS = ROOT / 'weights'
EXPORTS = ROOT / 'exports'

for p in (RAW, EXTRACTED, MERGED, WEIGHTS, EXPORTS):
    p.mkdir(parents=True, exist_ok=True)

print('Drive folder:', ROOT)
print('Old merged data.yaml exists:', (MERGED / 'data.yaml').exists())
print('OK — folders ready')

In [ ]:
!df -h /content | tail -n 1
print('Check Drive space: https://drive.google.com/drive/quota')
print('Download ONE dataset cell at a time. Delete a zip later if space gets tight.')

In [ ]:
def download(url: str, dest: Path, min_bytes: int = 1_000_000):
    """Download to Colab disk first, then copy to Drive.

    wget -O straight into Google Drive often dies with exit 8
    (S3 403 / FUSE timeout). curl + local /content is more reliable.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > min_bytes:
        print(f'Already exists ({dest.stat().st_size/1e6:.1f} MB): {dest.name}')
        return dest
    if dest.exists():
        print(f'Removing incomplete Drive file ({dest.stat().st_size} bytes)')
        dest.unlink()

    tmp = Path('/content') / dest.name
    if tmp.exists() and tmp.stat().st_size <= min_bytes:
        tmp.unlink()

    if not (tmp.exists() and tmp.stat().st_size > min_bytes):
        print('Downloading →', dest.name)
        curl = [
            'curl', '-L', '--fail', '--retry', '8', '--retry-all-errors',
            '--retry-delay', '3', '-C', '-',
            '-A', 'Mozilla/5.0 (compatible; RAASTA-colab/1.0)',
            '-o', str(tmp), url,
        ]
        try:
            subprocess.check_call(curl)
        except subprocess.CalledProcessError:
            print('curl failed, trying wget...')
            subprocess.check_call([
                'wget', '-c', '--tries=8', '--waitretry=3',
                '-U', 'Mozilla/5.0', '-O', str(tmp), url,
            ])
        if not tmp.exists() or tmp.stat().st_size < min_bytes:
            raise RuntimeError(f'Download too small: {tmp}')
        print(f'Done local: {tmp.stat().st_size/1e6:.1f} MB')

    print('Copying to Drive...')
    shutil.copy2(tmp, dest)
    print(f'Done: {dest.stat().st_size/1e6:.1f} MB')
    return dest

def download_gdrive(file_id: str, dest: Path, min_bytes: int = 1_000_000):
    import gdown
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > min_bytes:
        print(f'Already exists ({dest.stat().st_size/1e6:.1f} MB): {dest.name}')
        return dest
    print('Downloading from Google Drive →', dest.name)
    gdown.download(id=file_id, output=str(dest), quiet=False)
    if not dest.exists() or dest.stat().st_size < min_bytes:
        raise RuntimeError(f'Drive download looks too small: {dest}')
    print(f'Done: {dest.stat().st_size/1e6:.1f} MB')
    return dest

def unzip_to(zip_path: Path, out_dir: Path):
    """Unzip on Colab local disk. Drive FUSE is too slow for thousands of photos."""
    local_dir = Path('/content/extracted') / out_dir.name
    local_marker = local_dir / '_EXTRACT_OK'
    drive_marker = out_dir / '_EXTRACT_OK'
    if local_marker.exists():
        print('Already extracted locally:', local_dir)
        return local_dir
    if drive_marker.exists():
        print('Already extracted on Drive:', out_dir)
        return out_dir

    local_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting', zip_path.name, '→ local', local_dir)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(local_dir)
    local_marker.write_text('ok')
    print('Extract done locally')
    return local_dir

print('Helpers ready')

In [ ]:
!pip -q install ultralytics gdown roboflow pyyaml
print('Packages ready')

## 1) India roads — cracks and potholes (~0.5 GB)

Daytime dashcam holes and cracks. Run this first.
Source: [RDD2022 India](https://github.com/sekilab/RoadDamageDetector)

If this cell errors (`wget` exit 8), it is the AWS link, not your notebook. Re-run the helper cell, then this one. If it still fails, skip it and continue — the pothole cell still trains holes.

In [ ]:
INDIA_URL = (
    'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/'
    '2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip'
)
india_zip = RAW / 'RDD2022_India.zip'

# A leftover 0-byte zip from the failed wget will block retries.
for leftover in (india_zip, Path('/content/RDD2022_India.zip')):
    if leftover.exists() and leftover.stat().st_size < 1_000_000:
        leftover.unlink()
        print('Deleted incomplete', leftover)

try:
    download(INDIA_URL, india_zip, min_bytes=50_000_000)
    unzip_to(india_zip, EXTRACTED / 'RDD2022_India')
    get_ipython().system(
        'find /content/drive/MyDrive/raasta/extracted/RDD2022_India '
        '-maxdepth 3 -type d | head -n 40'
    )
    print('India ready')
except Exception as e:
    print('India download failed:', e)
    print(
        'SKIP India for now. That is OK. The next pothole cell still gives '
        'daytime holes. Later, download RDD2022_India.zip on your PC from '
        'https://github.com/sekilab/RoadDamageDetector and upload it to '
        'MyDrive/raasta/raw/'
    )

## 2) Extra pothole photos

Same zip used in the public pothole Colab tutorials. If Dropbox says the link is dead, skip this cell. India already has potholes.

In [ ]:
POTHOLE_URL = 'https://www.dropbox.com/s/qvglw8pqo16769f/pothole_dataset_v8.zip?dl=1'
try:
    p_zip = RAW / 'pothole_dataset_v8.zip'
    download(POTHOLE_URL, p_zip)
    unzip_to(p_zip, EXTRACTED / 'pothole_dataset_v8')
    print('Pothole zip ready')
except Exception as e:
    print('Pothole zip failed. Skip it and continue. Reason:', e)

## 3) Dark photos — people, dogs, cats (ExDark, ~1.5 GB)

These photos are taken in low light. We keep only **People, Dog, and Cat**. The other ExDark classes (bottle, chair, bus) are ignored.

Image file id and label file id are the official links from the [ExDark GitHub](https://github.com/cs-chan/Exclusively-Dark-Image-Dataset).

In [ ]:
try:
    # Images ~1.5 GB
    download_gdrive(
        '1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx',
        RAW / 'ExDark_images.zip',
        min_bytes=50_000_000,
    )
    unzip_to(RAW / 'ExDark_images.zip', EXTRACTED / 'ExDark')

    # Bounding-box labels ~4 MB
    download_gdrive(
        '1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i',
        RAW / 'ExDark_labels.zip',
        min_bytes=100_000,
    )
    unzip_to(RAW / 'ExDark_labels.zip', EXTRACTED / 'ExDark_Ann')
    print('ExDark ready')
except Exception as e:
    print('ExDark failed. Skip it and continue. Reason:', e)

## 4) Buffalo, horse, goat, and donkey

Run this before training. It downloads public photos only.

- **Buffalo** — African wildlife set (~100 MB). Elephant, rhino, and zebra boxes are thrown away.
- **Horse and goat** — Open Images. Up to 700 photos each.
- **Donkey** — Open Images has no donkey class. It does have **mule**. Those photos are saved as donkey, because a mule is the closest labeled animal.

The Open Images box file is about 1 GB. That is normal. Only horse, goat, and mule rows are kept.

In [ ]:
import csv
import urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

ANIMAL_OUT = EXTRACTED / 'animals_yolo'
if ANIMAL_OUT.exists():
    shutil.rmtree(ANIMAL_OUT)
for split in ('train', 'val'):
    (ANIMAL_OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (ANIMAL_OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

OUR_ID = {'buffalo': 5, 'horse': 8, 'donkey': 9, 'goat': 10}
animal_counts = {'buffalo': 0, 'horse': 0, 'donkey': 0, 'goat': 0}

def save_animal(img_src: Path, lines, stem: str):
    if not lines or not img_src.exists() or img_src.stat().st_size < 1000:
        return
    split = 'val' if (sum(stem.encode()) % 10 == 0) else 'train'
    dst = ANIMAL_OUT / split / 'images' / f'{stem}{img_src.suffix.lower()}'
    if not dst.exists():
        shutil.copy2(img_src, dst)
    (ANIMAL_OUT / split / 'labels' / f'{stem}.txt').write_text(chr(10).join(lines) + chr(10))
    for line in lines:
        cid = int(line.split()[0])
        for name, i in OUR_ID.items():
            if i == cid:
                animal_counts[name] += 1

# --- buffalo ---
download(
    'https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip',
    RAW / 'african-wildlife.zip',
    min_bytes=20_000_000,
)
unzip_to(RAW / 'african-wildlife.zip', EXTRACTED / 'african-wildlife')
wild_imgs = {}
for p in (EXTRACTED / 'african-wildlife').rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        wild_imgs.setdefault(p.stem, p)
for lbl in (EXTRACTED / 'african-wildlife').rglob('*.txt'):
    img = wild_imgs.get(lbl.stem)
    if img is None:
        continue
    lines = []
    for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split()
        if len(parts) >= 5 and int(float(parts[0])) == 0:
            parts[0] = '5'
            lines.append(' '.join(parts[:5]))
    save_animal(img, lines, 'buf_' + lbl.stem)
print('Buffalo boxes so far:', animal_counts['buffalo'])

# --- horse, goat, mule-as-donkey from Open Images ---
OI = {
    '/m/03k3r': 'horse',
    '/m/03fwl': 'goat',
    '/m/0dbzx': 'donkey',
}
CAP = 700
csv_path = RAW / 'oidv6-train-annotations-bbox.csv'
download(
    'https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv',
    csv_path,
    min_bytes=50_000_000,
)

chosen = {name: set() for name in ('horse', 'goat', 'donkey')}
boxes = defaultdict(list)
with csv_path.open(newline='', encoding='utf-8', errors='ignore') as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = OI.get(row.get('LabelName', ''))
        if name is None:
            continue
        if row.get('Confidence') not in ('1', '1.0'):
            continue
        if row.get('IsGroupOf') == '1' or row.get('IsDepiction') == '1':
            continue
        iid = row['ImageID']
        if iid not in chosen[name] and len(chosen[name]) >= CAP:
            continue
        chosen[name].add(iid)
        xmin, xmax = float(row['XMin']), float(row['XMax'])
        ymin, ymax = float(row['YMin']), float(row['YMax'])
        w, h = xmax - xmin, ymax - ymin
        if w < 0.02 or h < 0.02:
            continue
        cx, cy = xmin + w / 2, ymin + h / 2
        boxes[iid].append(f"{OUR_ID[name]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        if all(len(chosen[n]) >= CAP for n in chosen):
            break
print('Open Images images kept:', {n: len(chosen[n]) for n in chosen})

oi_dir = EXTRACTED / 'open_images_animals'
oi_dir.mkdir(parents=True, exist_ok=True)

def fetch_oi(iid):
    dest = oi_dir / f'{iid}.jpg'
    if dest.exists() and dest.stat().st_size > 1000:
        return iid, dest
    url = f'https://open-images-dataset.s3.amazonaws.com/train/{iid}.jpg'
    try:
        urllib.request.urlretrieve(url, dest)
        if dest.stat().st_size > 1000:
            return iid, dest
    except Exception:
        dest.unlink(missing_ok=True)
    return iid, None

ok = 0
with ThreadPoolExecutor(max_workers=16) as ex:
    futs = [ex.submit(fetch_oi, iid) for iid in boxes]
    for i, fut in enumerate(as_completed(futs), 1):
        iid, dest = fut.result()
        if dest is not None:
            save_animal(dest, boxes[iid], 'oi_' + iid)
            ok += 1
        if i % 200 == 0 or i == len(futs):
            print(f'  photos {i}/{len(futs)} saved {ok}')

(ANIMAL_OUT / 'data.yaml').write_text(
    'path: ' + str(ANIMAL_OUT) + chr(10)
    + 'train: train/images' + chr(10)
    + 'val: val/images' + chr(10)
    + 'names:' + chr(10)
    + ''.join(f'  {i}: {n}{chr(10)}' for i, n in enumerate([
        'pothole', 'crack', 'speed_bump', 'person', 'cow', 'buffalo',
        'dog', 'cat', 'horse', 'donkey', 'goat',
    ]))
)
print('Animal boxes:', animal_counts)
print('Saved in', ANIMAL_OUT)

## 5) Optional — Japan roads (~1 GB)

More daytime cracks. **Leave `RUN_JAPAN = False`** unless Drive still has several GB free. This does not add night photos.

In [ ]:
RUN_JAPAN = False  # set True only if you have space

if not RUN_JAPAN:
    print('Skipped Japan. That is fine.')
else:
    JAPAN_URL = (
        'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/'
        '2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Japan.zip'
    )
    download(JAPAN_URL, RAW / 'RDD2022_Japan.zip')
    unzip_to(RAW / 'RDD2022_Japan.zip', EXTRACTED / 'RDD2022_Japan')
    print('Japan ready')

## 5b) Optional — Roboflow night people

Leave the key **blank** to skip. The rest of the notebook still trains.

If you want more night pedestrians: open [Roboflow Universe](https://universe.roboflow.com), search `night pedestrian`, copy workspace / project / version into `NIGHT_JOBS` below, and paste your API key (Roboflow → Settings → API key).

In [ ]:
ROBOFLOW_API_KEY = ""  # paste key or leave blank

# (our class, workspace, project, version, folder name)
NIGHT_JOBS = [
    # ("person", "workspace-name", "project-name", 1, "rf_night_person"),
]

RF_DOWNLOADS = []

if not ROBOFLOW_API_KEY.strip() or not NIGHT_JOBS:
    print('Roboflow skipped. ExDark still covers dark people, dogs, and cats.')
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())
    rf_root = Path('/content/rf_raw')
    rf_root.mkdir(parents=True, exist_ok=True)
    for canon_name, ws, proj, ver, folder in NIGHT_JOBS:
        try:
            print(f'Downloading {ws}/{proj} v{ver}')
            ds = rf.workspace(ws).project(proj).version(ver).download(
                'yolov8', location=str(rf_root / folder)
            )
            RF_DOWNLOADS.append((canon_name, Path(ds.location)))
            print('  ok', ds.location)
        except Exception as e:
            print(f'  skip {ws}/{proj}:', e)
    print('Roboflow sets:', len(RF_DOWNLOADS))

## 6) Build the training folder

This cell turns every download into the same 11-class YOLO labels, including buffalo, horse, goat, and donkey, then copies about 2000 old photos from `MyDrive/raasta/merged`.

Unknown labels are **dropped**. They are not turned into potholes.

The mix is saved on the Colab disk (`/content/night_mix`), which is faster to train from. If Colab disconnects before training finishes, run this cell again, then the train cell.

In [ ]:
from pathlib import Path
import os, random, shutil, xml.etree.ElementTree as ET
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import yaml
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

ROOT = Path('/content/drive/MyDrive/raasta')
EXTRACTED = ROOT / 'extracted'
MERGED = ROOT / 'merged'
LOCAL_EXTRACTED = Path('/content/extracted')
print('Using', ROOT, 'Drive mounted:', ROOT.exists())

def data_dir(name: str) -> Path:
    local = LOCAL_EXTRACTED / name
    drive = EXTRACTED / name
    if local.exists():
        return local
    return drive

if 'RF_DOWNLOADS' not in globals():
    RF_DOWNLOADS = []

CLASS_NAMES = [
    'pothole', 'crack', 'speed_bump', 'person', 'cow', 'buffalo',
    'dog', 'cat', 'horse', 'donkey', 'goat',
]
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
ALIASES = {
    'pothole': 'pothole', 'potholes': 'pothole', 'pot hole': 'pothole',
    'd40': 'pothole', 'hole': 'pothole',
    'crack': 'crack', 'cracks': 'crack', 'd00': 'crack', 'd10': 'crack', 'd20': 'crack',
    'longitudinal crack': 'crack', 'transverse crack': 'crack', 'alligator crack': 'crack',
    'speed_bump': 'speed_bump', 'speed bump': 'speed_bump', 'speedbump': 'speed_bump',
    'bump': 'speed_bump', 'hump': 'speed_bump', 'speed hump': 'speed_bump',
    'person': 'person', 'people': 'person', 'pedestrian': 'person', 'human': 'person',
    'cow': 'cow', 'cattle': 'cow', 'buffalo': 'buffalo',
    'dog': 'dog', 'cat': 'cat', 'horse': 'horse', 'donkey': 'donkey', 'goat': 'goat',
}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
rng = random.Random(42)

def canon(raw: str):
    key = (raw or '').strip().lower().replace('_', ' ').replace('-', ' ')
    key = ' '.join(key.split())
    return ALIASES.get(key)

OUT = Path('/content/night_mix')
if OUT.exists():
    shutil.rmtree(OUT)
for split in ('train', 'val'):
    (OUT / 'images' / split).mkdir(parents=True)
    (OUT / 'labels' / split).mkdir(parents=True)

box_counts = Counter()
file_counts = {'train': 0, 'val': 0}

def add_sample(img: Path, lines, split: str, prefix: str):
    if not lines:
        return
    stem = f'{prefix}_{file_counts[split]}_{img.stem}'[:180]
    dst = OUT / 'images' / split / f'{stem}{img.suffix.lower()}'
    try:
        os.link(img, dst)
    except Exception:
        shutil.copy2(img, dst)
    (OUT / 'labels' / split / f'{stem}.txt').write_text(chr(10).join(lines) + chr(10))
    file_counts[split] += 1
    for line in lines:
        box_counts[CLASS_NAMES[int(line.split()[0])]] += 1

def yolo_line(cid, cx, cy, w, h):
    if w <= 0 or h <= 0:
        return None
    cx, cy, w, h = [max(0.0, min(1.0, v)) for v in (cx, cy, w, h)]
    if w < 0.002 or h < 0.002:
        return None
    return f'{cid} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'

def index_images(root: Path):
    idx = {}
    if not root.exists():
        return idx
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXT and p.name != 'Thumbnails.png':
            idx.setdefault(p.stem, p)
    return idx

# ----- A) old merged set: keep road skill -----
def read_old(img: Path, lbl_dir: Path):
    lbl = lbl_dir / f'{img.stem}.txt'
    if not lbl.exists():
        return None
    clean = []
    for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split()
        if len(parts) >= 5 and int(float(parts[0])) < len(CLASS_NAMES):
            clean.append(' '.join(parts[:5]))
    return clean or None

if (MERGED / 'images' / 'train').exists():
    for split, limit in (('train', 2000), ('val', 300)):
        img_dir = MERGED / 'images' / split
        lbl_dir = MERGED / 'labels' / split
        if not img_dir.exists() or not lbl_dir.exists():
            print('skip old', split, '(folder missing)')
            continue
        imgs = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT]
        rng.shuffle(imgs)
        imgs = imgs[:limit]
        print(f'Old merged {split}: copying {len(imgs)}')
        jobs = []
        with ThreadPoolExecutor(max_workers=8) as ex:
            futs = [ex.submit(read_old, img, lbl_dir) for img in imgs]
            for img, fut in zip(imgs, futs):
                lines = fut.result()
                if lines:
                    jobs.append((img, lines))
        for img, lines in jobs:
            add_sample(img, lines, split, 'old')
else:
    print('WARNING: MyDrive/raasta/merged/images/train not found. Old road skill will not be kept.')

print('After old set:', dict(box_counts), 'files', file_counts)

# ----- B) RDD xml (India, and Japan if you downloaded it) -----
def rdd_lines(xml_path: Path, img_path: Path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    W = H = None
    if size is not None and size.find('width') is not None:
        W = float(size.find('width').text)
        H = float(size.find('height').text)
    if not W or not H:
        with Image.open(img_path) as im:
            W, H = im.size
    lines = []
    for obj in root.findall('object'):
        name = obj.findtext('name') or ''
        cname = canon(name)
        if cname is None:
            continue
        bb = obj.find('bndbox')
        if bb is None:
            continue
        xmin = float(bb.findtext('xmin'))
        ymin = float(bb.findtext('ymin'))
        xmax = float(bb.findtext('xmax'))
        ymax = float(bb.findtext('ymax'))
        line = yolo_line(
            CLASS_TO_ID[cname],
            ((xmin + xmax) / 2) / W,
            ((ymin + ymax) / 2) / H,
            (xmax - xmin) / W,
            (ymax - ymin) / H,
        )
        if line:
            lines.append(line)
    return lines

RDD_CAP = 2500
for folder_name in ('RDD2022_India', 'RDD2022_Japan'):
    root = data_dir(folder_name)
    if not root.exists():
        print('skip', folder_name)
        continue
    print('Indexing', folder_name, '...')
    images = index_images(root)
    xmls = [p for p in root.rglob('*.xml')]
    rng.shuffle(xmls)
    kept = 0
    for xml_path in xmls:
        if kept >= RDD_CAP:
            break
        img = images.get(xml_path.stem)
        if img is None:
            continue
        try:
            lines = rdd_lines(xml_path, img)
        except Exception:
            continue
        split = 'val' if kept % 10 == 0 else 'train'
        before = file_counts[split]
        add_sample(img, lines, split, 'rdd')
        kept += file_counts[split] - before
    print(folder_name, 'added', kept)

# ----- C) pothole YOLO zip -----
def read_names(yaml_path: Path):
    if not yaml_path.exists():
        return None
    data = yaml.safe_load(yaml_path.read_text(encoding='utf-8', errors='ignore')) or {}
    names = data.get('names')
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    if isinstance(names, list):
        return {i: str(n) for i, n in enumerate(names)}
    return None

def yolo_pairs(root: Path):
    pairs = []
    layouts = [
        ('train/images', 'train/labels'),
        ('valid/images', 'valid/labels'),
        ('val/images', 'val/labels'),
        ('images/train', 'labels/train'),
        ('images/val', 'labels/val'),
    ]
    # also search one level down (zip often has one wrapper folder)
    roots = [root] + [p for p in root.iterdir() if p.is_dir()] if root.exists() else []
    seen = set()
    for base in roots:
        for rel_i, rel_l in layouts:
            img_dir, lbl_dir = base / rel_i, base / rel_l
            if not img_dir.is_dir() or not lbl_dir.is_dir():
                continue
            for img in img_dir.iterdir():
                if not img.is_file() or img.suffix.lower() not in IMG_EXT:
                    continue
                lbl = lbl_dir / f'{img.stem}.txt'
                if lbl.exists() and img not in seen:
                    seen.add(img)
                    pairs.append((img, lbl, base))
    return pairs

p_root = data_dir('pothole_dataset_v8')
pairs = yolo_pairs(p_root)
rng.shuffle(pairs)
added = 0
for img, lbl, base in pairs[:1500]:
    names = read_names(base / 'data.yaml') or {0: 'pothole'}
    lines = []
    for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        raw = names.get(int(float(parts[0])), '')
        cname = canon(raw)
        if cname is None:
            continue
        parts[0] = str(CLASS_TO_ID[cname])
        lines.append(' '.join(parts[:5]))
    split = 'val' if added % 10 == 0 else 'train'
    before = file_counts[split]
    add_sample(img, lines, split, 'pot')
    added += file_counts[split] - before
print('Pothole zip added', added)

# ----- C2) buffalo, horse, goat, donkey -----
animal_root = data_dir('animals_yolo')
animal_pairs = yolo_pairs(animal_root)
print('Animal photos found:', len(animal_pairs))
added = 0
for img, lbl, base in animal_pairs:
    names = read_names(base / 'data.yaml') or read_names(animal_root / 'data.yaml') or {}
    lines = []
    for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        raw = names.get(int(float(parts[0])), '')
        cname = canon(raw)
        if cname is None:
            continue
        parts[0] = str(CLASS_TO_ID[cname])
        lines.append(' '.join(parts[:5]))
    split = 'val' if added % 10 == 0 else 'train'
    before = file_counts[split]
    add_sample(img, lines, split, 'ani')
    added += file_counts[split] - before
print('Buffalo / horse / goat / donkey added', added)

# ----- D) ExDark dark people / dog / cat -----
KEEP_EXDARK = {'people', 'person', 'dog', 'cat'}
ex_imgs = index_images(data_dir('ExDark'))
ex_txts = []
ann_root = data_dir('ExDark_Ann')
if ann_root.exists():
    ex_txts = [p for p in ann_root.rglob('*.txt') if p.name.lower() != 'imageclasslist.txt']
print('ExDark images indexed', len(ex_imgs), 'label files', len(ex_txts))
added = 0
for txt in ex_txts:
    img = ex_imgs.get(txt.stem)
    if img is None:
        continue
    try:
        with Image.open(img) as im:
            W, H = im.size
    except Exception:
        continue
    lines = []
    for raw_line in txt.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = raw_line.split()
        cname = None
        nums = None
        for i, tok in enumerate(parts):
            cname = canon(tok) if tok.lower() in KEEP_EXDARK or canon(tok) in ('person', 'dog', 'cat') else None
            if cname in ('person', 'dog', 'cat') and i + 4 < len(parts):
                try:
                    l, t, w, h = map(float, parts[i + 1:i + 5])
                except ValueError:
                    cname = None
                    continue
                nums = (l, t, w, h)
                break
            cname = None
        if not nums or nums[2] <= 1 or nums[3] <= 1:
            continue
        l, t, w, h = nums
        line = yolo_line(CLASS_TO_ID[cname], (l + w / 2) / W, (t + h / 2) / H, w / W, h / H)
        if line:
            lines.append(line)
    split = 'val' if added % 10 == 0 else 'train'
    before = file_counts[split]
    add_sample(img, lines, split, 'dark')
    added += file_counts[split] - before
print('ExDark added', added)

# ----- E) optional Roboflow -----
for force_name, root in RF_DOWNLOADS:
    names = read_names(root / 'data.yaml')
    got = 0
    for img, lbl, base in yolo_pairs(root)[:1200]:
        lines = []
        for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            if force_name in CLASS_TO_ID:
                cname = force_name
            else:
                raw = (names or {}).get(int(float(parts[0])), '')
                cname = canon(raw)
            if cname is None:
                continue
            parts[0] = str(CLASS_TO_ID[cname])
            lines.append(' '.join(parts[:5]))
        split = 'val' if got % 10 == 0 else 'train'
        before = file_counts[split]
        add_sample(img, lines, split, 'rf')
        got += file_counts[split] - before
    print('Roboflow', force_name, 'added', got)

if file_counts['val'] < 20 and file_counts['train'] > 50:
    train_imgs = list((OUT / 'images' / 'train').iterdir())
    rng.shuffle(train_imgs)
    move_n = max(20, len(train_imgs) // 10)
    for img in train_imgs[:move_n]:
        lbl = OUT / 'labels' / 'train' / f'{img.stem}.txt'
        img.rename(OUT / 'images' / 'val' / img.name)
        if lbl.exists():
            lbl.rename(OUT / 'labels' / 'val' / lbl.name)
        file_counts['train'] -= 1
        file_counts['val'] += 1

(OUT / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(OUT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}))
print('----')
print('Files:', file_counts)
print('Boxes:', dict(box_counts))
print('data.yaml:', OUT / 'data.yaml')
assert file_counts['train'] > 100, 'Too few training images. Check the download cells.'

## 7) Train — 30 epochs from your latest best.pt

Runtime must be **T4 GPU**.

If the cell says CUDA out of memory, change `batch=8` to `batch=4` and run it again.

If Colab disconnects in the middle, run the **resume** cell under this one. Do not start from scratch.

In [ ]:
from pathlib import Path
import gc
import torch
from ultralytics import YOLO

ROOT = Path('/content/drive/MyDrive/raasta')
WEIGHTS = ROOT / 'weights'
OUT = Path('/content/night_mix')
assert (OUT / 'data.yaml').exists(), 'Run the mix cell first so /content/night_mix/data.yaml exists.'

CANDIDATES = [
    WEIGHTS / 'm2_m5_people_animals_v1/finetune/weights/best.pt',
    WEIGHTS / 'm2_m5_ft1/finetune/weights/best.pt',
    WEIGHTS / 'm2_m5_v1/weights/best.pt',
    WEIGHTS / 'm2_m5_v1/finetune/weights/best.pt',
]
BEST = next((p for p in CANDIDATES if p.exists()), None)
if BEST is None:
    found = sorted(WEIGHTS.rglob('best.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    BEST = found[0] if found else None
assert BEST is not None, 'No best.pt under MyDrive/raasta/weights. Upload your last weights first.'
print('Starting from:', BEST)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime → Change runtime type → T4, then run this cell again.')

RUN = WEIGHTS / 'm2_m5_night_public_v1'
RUN.mkdir(parents=True, exist_ok=True)

model = YOLO(str(BEST))
model.train(
    data=str(OUT / 'data.yaml'),
    epochs=30,
    imgsz=640,
    batch=8,
    workers=2,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(RUN),
    name='finetune',
    exist_ok=True,
    pretrained=True,
    lr0=0.0006,
    lrf=0.01,
    patience=12,
    cache=False,
    amp=True,
    plots=True,
    save_period=5,
    mosaic=0.5,
    close_mosaic=8,
    fliplr=0.5,
    degrees=8.0,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.45,
)
FT_BEST = RUN / 'finetune' / 'weights' / 'best.pt'
print('New best:', FT_BEST)

### Resume if Colab stopped mid-train
Run setup (cell 0) first so `WEIGHTS` exists, then this cell.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = Path('/content/drive/MyDrive/raasta/weights')
last = WEIGHTS / 'm2_m5_night_public_v1/finetune/weights/last.pt'
print('Resume file exists:', last.exists(), last)
# Uncomment the next 2 lines only if training stopped halfway:
# model = YOLO(str(last))
# model.train(resume=True)

## 8) Export the phone file (image size 640)

This writes:
`MyDrive/raasta/exports/raasta_m2_m5_night.tflite`

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil

ROOT = Path('/content/drive/MyDrive/raasta')
WEIGHTS = ROOT / 'weights'
EXPORTS = ROOT / 'exports'
EXPORTS.mkdir(parents=True, exist_ok=True)

FT_BEST = WEIGHTS / 'm2_m5_night_public_v1/finetune/weights/best.pt'
assert FT_BEST.exists(), FT_BEST
model = YOLO(str(FT_BEST))
export_path = model.export(format='tflite', imgsz=640, int8=False)
print('TFLite:', export_path)

shutil.copy2(FT_BEST, EXPORTS / 'raasta_m2_m5_night.pt')
shutil.copy2(export_path, EXPORTS / 'raasta_m2_m5_night.tflite')
print('Copied to', EXPORTS / 'raasta_m2_m5_night.tflite')

## 9) Put it in the app (on your PC)

1. In Google Drive, download `MyDrive/raasta/exports/raasta_m2_m5_night.tflite`
2. Replace this file (same name the app already loads):

`E:\\Android\\projects\\raasta_app\\assets\\models\\raasta_m2_m5.tflite`

3. Run the app again. Class count stays 11. Do not change the class order.

The phone will still miss some far or tiny holes. This training pass is what makes close holes, people, and animals show up more often in day and in the dark.

## Optional — delete a zip after it extracted
Only after the extract folder looks right. This frees Drive space.

In [ ]:
# (RAW / 'RDD2022_India.zip').unlink(missing_ok=True)
# (RAW / 'pothole_dataset_v8.zip').unlink(missing_ok=True)
# (RAW / 'ExDark_images.zip').unlink(missing_ok=True)
print('Left commented on purpose. Uncomment a line only after that dataset extracted.')